# Explainable AI (XAI) - Unified Mistake Analysis

This notebook provides a detailed analysis of grading mistakes using the same services as the evaluation script. It helps understand why certain answers are misclassified and provides insights into the grading system's behavior.

## Overview

This analysis examines:
- How correct answers are classified
- How incorrect answers are classified
- Specific mistakes (False Positives and False Negatives)
- Key points hit and missed by each answer


## Setup and Imports


In [1]:
import os
import sys
from dotenv import load_dotenv
import json
from pathlib import Path

if os.getcwd().endswith('notebooks'):
    root_dir = Path(os.getcwd()).parent
else:
    root_dir = Path(os.getcwd())

backend_dir = root_dir / "backend"

sys.path.insert(0, str(backend_dir))

In [2]:

from services.grading_service import GradingService
from services.question_service import QuestionService
import openai

env_path = root_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv()  # Try default location

✅ Successfully loaded configuration from C:\Users\V\Desktop\llm7\backend\core\..\settings.yaml


## Unified Mistake Analysis Class

This class uses the **EXACT same logic** as the evaluation script to ensure consistency. It analyzes how answers are graded and identifies misclassifications.


In [3]:
class UnifiedMistakeAnalysis:
    
    def __init__(self):
        self._load_questions_from_data_files()
        if len(self._questions) == 0:
            print("No questions loaded")
            return
        self._initialize_grading_service()
        
    def _load_questions_from_data_files(self) -> None:
        annotated_file = backend_dir / "evaluation" / "annotated_questions.json"
        self._questions = []
        
        if annotated_file.exists():
            with open(annotated_file, 'r', encoding='utf-8') as f:
                self._questions = json.load(f)
                print(f"Loaded {len(self._questions)} annotated questions with realistic student answers")
        else:
            print(f"Annotated questions file not found: {annotated_file}")
    
    def _initialize_grading_service(self) -> None:
        """Initialize grading service - exactly the same as evaluation_script.py"""
        openai_api_key = os.getenv("OPENAI_API_KEY")
        if not openai_api_key:
            raise ValueError("OPENAI_API_KEY environment variable not set")
        
        print(f"✅ OpenAI API key found")
        openai.api_key = openai_api_key
        
        # Creates question service with our loaded questions - EXACT same as evaluation_script.py
        self._question_service = QuestionService()
        # Overrides with our annotated questions data
        self._question_service._questions_bank = self._questions  # Uses _questions_bank instead of _questions
        self._question_service._questions_by_id = {q["question_id"]: q for q in self._questions}
        
        #  !Marks as loaded to prevent reloading fallback questions
        self._question_service._is_loaded = True
        
        print(f"Loaded {len(self._questions)} questions for evaluation")
        for q in self._questions[:3]:  # Show first 3
            print(f"  Question {q['question_id']}: {q['question_text'][:50]}...")
            print(f"    Key Points: {[kp['text'][:30] + '...' for kp in q['key_points']]}")
        
        # Creates evaluation service - EXACT same as evaluation_script.py
        self._grading_service = GradingService(
            self._question_service, 
            openai.OpenAI(api_key=openai_api_key)
        )
        self._grading_service.precompute_embeddings()
        
        print(f"Grading service initialized with correct questions")

    def analyze_detailed_mistakes(self):
        
        print("🔍 DETAILED MISTAKE ANALYSIS")
        print("=" * 80)
        
        for question in self._questions[:3]:  # First 3 questions only for detailed analysis
            self.analyze_question_detailed(question)

    def analyze_question_detailed(self, question):       
        question_id = question["question_id"]
        question_text = question["question_text"]
        
        print(f"\n📝 QUESTION {question_id}: {question_text}")
        print("=" * 80)
        
        print(f"Key Points:")
        for i, kp in enumerate(question["key_points"]):
            print(f"  {i+1}. {kp['text']}")
        
        print(f"\nClassification threshold: 50% (answers >= 50% score are considered correct)")
        
        # Analyze correct answers
        print(f"\n✅ ANALYZING CORRECT ANSWERS:")
        print("-" * 60)
        
        correct_correctly_classified = 0
        correct_incorrectly_classified = 0
        
        for i, answer in enumerate(question["correct_answers"]):
            print(f"\n  Correct Answer {i+1}:")
            print(f"  Text: '{answer}'")
            
            try:
                result = self._grading_service.grade_answer(question_id, answer)
                score = result.score
                is_correct = score >= 50.0  # Classification threshold
                
                status = "✅ CORRECTLY CLASSIFIED" if is_correct else "❌ INCORRECTLY CLASSIFIED"
                print(f"  Score: {score}%")
                print(f"  Classification: {status}")
                print(f"  Hit Key Points: {result.hit_key_points}")
                print(f"  Missing Key Points: {result.missing_key_points}")
                
                if is_correct:
                    correct_correctly_classified += 1
                else:
                    correct_incorrectly_classified += 1
                    print(f"  🚨 MISTAKE: Correct answer marked as incorrect!")
                
            except Exception as e:
                print(f"  Error: {e}")
                correct_incorrectly_classified += 1
        
        # Analyzes incorrect answers
        print(f"\n❌ ANALYZING INCORRECT ANSWERS:")
        print("-" * 60)
        
        incorrect_correctly_classified = 0
        incorrect_incorrectly_classified = 0
        
        for i, answer in enumerate(question["incorrect_answers"]):
            print(f"\n  Incorrect Answer {i+1}:")
            print(f"  Text: '{answer}'")
            
            try:
                result = self._grading_service.grade_answer(question_id, answer)
                score = result.score
                is_correct = score >= 50.0  # Classification threshold
                
                status = "✅ CORRECTLY CLASSIFIED" if not is_correct else "❌ INCORRECTLY CLASSIFIED"
                print(f"  Score: {score}%")
                print(f"  Classification: {status}")
                print(f"  Hit Key Points: {result.hit_key_points}")
                print(f"  Missing Key Points: {result.missing_key_points}")
                
                if not is_correct:
                    incorrect_correctly_classified += 1
                else:
                    incorrect_incorrectly_classified += 1
                    print(f"  🚨 MISTAKE: Incorrect answer marked as correct!")
                
            except Exception as e:
                print(f"  Error: {e}")
                incorrect_incorrectly_classified += 1
        
        total_correct_answers = len(question["correct_answers"])
        total_incorrect_answers = len(question["incorrect_answers"])
        total_answers = total_correct_answers + total_incorrect_answers
        
        total_correctly_classified = correct_correctly_classified + incorrect_correctly_classified
        accuracy = (total_correctly_classified / total_answers) * 100 if total_answers > 0 else 0
        
        print(f"\nQUESTION {question_id} SUMMARY:")
        print(f"  Correct Answers: {correct_correctly_classified}/{total_correct_answers} correctly classified")
        print(f"  Incorrect Answers: {incorrect_correctly_classified}/{total_incorrect_answers} correctly classified")
        print(f"  Overall Accuracy: {total_correctly_classified}/{total_answers} ({accuracy:.1f}%)")
        
        if correct_incorrectly_classified > 0 or incorrect_incorrectly_classified > 0:
            print(f"\n🚨 MISTAKES FOUND:")
            if correct_incorrectly_classified > 0:
                print(f"  - {correct_incorrectly_classified} correct answers marked as incorrect (False Negatives)")
            if incorrect_incorrectly_classified > 0:
                print(f"  - {incorrect_incorrectly_classified} incorrect answers marked as correct (False Positives)")


## Initialize Analysis

Create an instance of the UnifiedMistakeAnalysis class. This will:
1. Load questions from the annotated questions file
2. Initialize the grading service with OpenAI
3. Precompute embeddings for efficient analysis


In [ ]:
analysis = UnifiedMistakeAnalysis()


Loaded 8 annotated questions with realistic student answers
✅ OpenAI API key found
Loaded 8 questions for evaluation
  Question 1: Explain the role of erosion in shaping landscapes....
    Key Points: ['Erosion removes soil and rock ...', 'Water, wind, and ice are erosi...', 'Erosion creates valleys and ca...', 'Sediment transport alters land...']
  Question 2: What is natural selection and how does it contribu...
    Key Points: ['Survival of the fittest organi...', 'Variation in traits among indi...', 'Environment influences which t...', 'Successful traits are passed t...']
  Question 3: How do plants make their own food?...
    Key Points: ['Use sunlight as energy...', 'Take in carbon dioxide...']
🔄 Attempting to load embeddings from cache...
✅ Loaded embeddings cache from C:\Users\V\Desktop\llm7\backend\services\..\embeddings_cache.json
   📅 Created: 2025-10-30T20:48:16.633375
   🤖 Model: text-embedding-ada-002
   📊 8 questions, 26 embeddings
✅ Loaded embeddings from cache for 8 qu

## Run Detailed Mistake Analysis

This will analyze the first 3 questions in detail, showing:
- Each correct answer and how it was classified
- Each incorrect answer and how it was classified
- Key points hit and missed
- Specific mistakes (False Positives and False Negatives)


In [ ]:
if len(analysis._questions) > 0:
    analysis.analyze_detailed_mistakes()
else:
    print("❌ No questions loaded, cannot proceed with analysis")


🔍 DETAILED MISTAKE ANALYSIS

📝 QUESTION 1: Explain the role of erosion in shaping landscapes.
Key Points:
  1. Erosion removes soil and rock materials.
  2. Water, wind, and ice are erosion agents.
  3. Erosion creates valleys and canyons over time.
  4. Sediment transport alters landforms and ecosystems.

Classification threshold: 50% (answers >= 50% score are considered correct)

✅ ANALYZING CORRECT ANSWERS:
------------------------------------------------------------

  Correct Answer 1:
  Text: 'Erosion is when water and wind wear away rocks and soil. Over many years this creates valleys and canyons. The eroded stuff gets moved around and changes the landscape.'
  📊 Key point 'Erosion removes soil and rock ...': sim=0.933, overlap=0.60, hit=True
  📊 Key point 'Water, wind, and ice are erosi...': sim=0.884, overlap=0.60, hit=True
  📊 Key point 'Erosion creates valleys and ca...': sim=0.930, overlap=0.80, hit=True
  📊 Key point 'Sediment transport alters land...': sim=0.882, overlap=